In [0]:
from pyspark.sql import functions as F
catalog = spark.sql(
     "SELECT current_catalog()"
).first()[0]

schema = "migration"
df_customers = spark.table(f"{catalog}.{schema}.customers_bronze")
df_orders = spark.table(f"{catalog}.{schema}.orders_bronze")
df_products = spark.table(f"{catalog}.{schema}.products_bronze")
df_transactions = spark.table(f"{catalog}.{schema}.transactions_bronze")

In [0]:
#Customer Transformation 

#step 1: Remove uneccessary spaces 
text_columns=["customer_id","full_name","email","phone","registration_date","account_type","loyalty_tier","marketing_opt_in"]

for c in text_columns:
    df_customers = df_customers.withColumn(c, F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.trim(F.col(c))))

# step 2: remove ID prefixes
df_customers= df_customers.withColumn("customer_id_cleaned", F.regexp_replace("customer_id", r"^[A-Za-z]", ""))

improper_id= df_customers.filter(~F.col("customer_id_cleaned").rlike(r"^\d+$")).count()
distinct_ids_before = df_customers.select("customer_id").distinct().count()
distinct_ids_after = df_customers.select("customer_id_cleaned").distinct().count()

assert improper_id == 0,"Some IDs are not digits only"
assert distinct_ids_before == distinct_ids_after,"Removing the prefix made two different customer IDs identical."

df_customers = df_customers.drop("customer_id").withColumnRenamed("customer_id_cleaned","customer_id")

# step 3: casing
df_customers = (df_customers.withColumn("full_name", F.initcap("full_name"))
                .withColumn("email",F.lower("email"))
                .withColumn ("account_type",F.initcap("account_type"))
                .withColumn ("loyalty_tier",F.when(F.lower("loyalty_tier")=="none", F.lit(None))
                .otherwise(F.initcap("loyalty_tier")),)
            )

parts = F.split("full_name", "", 2)
df_customers = df_customers.withColumn("first_name", parts.getItem(0)).withColumn("last_name",parts.getItem(1))

# step4: dates
date = F.col("registration_date")
df_customers = df_customers.withColumn("registration_date",F.when(date.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(date,"yyyy-MM-dd"))
                                       .when(date.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(date,"dd/MM/yyyy"))
                                       .when(date.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(date,"MM-dd-yyyy")), 
                                       )

print(df_customers.filter(F.col("registration_date").isNull()).count())

#Step 5: booleans
df_customers = df_customers.withColumn("marketing_opt_in", F.when(F.lower("marketing_opt_in").isin("y","1","true"), True)
                                       .when(F.lower("marketing_opt_in").isin("n","0","false"), False),)

df_customers.groupBy("marketing_opt_in").count().show()

#step 6: remove duplicates
meta = ["ingested_at","source_file","batch_id"]
cols = [c for c in df_customers.columns if c not in meta]

print(df_customers.count())
duplicates_by_key =df_customers.dropDuplicates(["customer_id"]).count()
duplicates_by_row =df_customers.dropDuplicates(cols).count()

print("duplicates by key:", duplicates_by_key)
print("duplcates by row:", duplicates_by_row)
assert duplicates_by_key == duplicates_by_row, "Some customers have copies that still differ"

df_customers= df_customers.dropDuplicates(["customer_id"])

df_customers.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.customers_silver")


In [0]:
#Orders Transformation


In [0]:
#Products Trandformation
#step 1: removing unesseccary spaces
text_columns=["product_id","sku","product_name","category","sub_category","description","active_flag"]

for c in text_columns:
    df_products = df_products.withColumn(c, F.when(F.trim(F.col(c)) == "",F.lit(None)).otherwise(F.trim(F.col(c))))

#step 2: fix IDs
df_products =Df_products.withColumn()

In [0]:
#Transactions Transformation